# llm-gmc — the GPU half

Loads `cpu_members.npz` from the `ntl-cpu` kernel — SVC, tfidf_lr, distort_lr and the out-of-fold
diagnostic already computed, for free — and does only the work that needs a GPU: the two Qwen LoRA
members, DeBERTa, and q14v5, whose head is label-permuted and gets reordered at run time from an
in-sample confusion matrix. That fix is what took 0.627 to 0.639.

Blend, per-class offsets from the OOF, damped non-uniform Sinkhorn, submit.

In [ ]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# ---------------------------------------------------------------------------
# CONFIG — every tunable in one place, each with where it came from.
# Nothing below is a guess: weights and thresholds are copied from a published notebook or set by a
# measurement recorded in the comment.
# ---------------------------------------------------------------------------
CONFIG = {
    # member weights, in log-probability space
    "w_svc": 1.5,           # every >=0.63 public notebook runs the 4-view SVC at 1.5
    "w_deberta": 0.25,      # part-two write-up
    "w_q14v5": 3.0,         # the largest member of the 0.654/0.660 blend
    "w_tfidf_lr": 0.25,     # 0.654 recipe
    "w_distort_lr": 0.10,   # 0.654 recipe
    # a member only votes if its measured pool accuracy clears this floor, then scales linearly
    "admit_floor": 0.40,
    "admit_scale": 0.20,
    # centred twin: subtract the test-cluster mean from a member's own log-probs
    "twin": dict(thresh=0.85, min_size=2, max_size=18),
    # adaptive SVC topic correction (the 0.639 stack's mechanism); lam=0 reproduces v42 exactly
    "correction": dict(thresh=0.95, min_size=8, max_size=18, lam=0.62),
    # final balancing: damped Sinkhorn toward 0.65*predicted marginal + 0.35*train prior
    "sinkhorn": dict(alpha=0.45, target_mix=0.35, iters=100),
    # UltraBERT never earned a vote (measured 0.2790 and 0.2965 on pool rows)
    "ultra_max_w": 0.30,
}
W_SVC = CONFIG["w_svc"]
W_DEBERTA = CONFIG["w_deberta"]
LORAS = {
    "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8,
                            base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8,
                            base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
TWIN = CONFIG["twin"]
CORRECTION = CONFIG["correction"]

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)
LABEL2I = {l: i for i, l in enumerate(LABELS)}


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map(LABEL2I).values
LOGIT = np.zeros((len(test), K))
REPORT = {}
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

In [ ]:
"""The data the GPU half needs, without the vectorisers.

The GPU notebook used to get `pool`, `train_texts`, `test_texts` and `y` from the cell that also fitted
the SVC. That cell is gone — the SVC arrives precomputed from the CPU half — so these lines replace it.
No TF-IDF, no fitting: seconds, not 28 minutes.
"""
import numpy as np
import pandas as pd


def _load_pool():
    """Prefer the biggest passage table that has text and model, ignoring the 75k pool_v4 itself."""
    import glob as _glob
    best, best_n, best_path = None, 0, None
    for path in _glob.glob(f"{INPUT_DIR}/**/*.parquet", recursive=True) + \
                _glob.glob(f"{INPUT_DIR}/**/*.csv", recursive=True):
        if "pool_v4" in path or "train.csv" in path or "test.csv" in path or "cpu_members" in path:
            continue
        try:
            df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
        except Exception:
            continue
        if not {"text", "model"} <= set(df.columns) or len(df) <= best_n:
            continue
        best, best_n, best_path = df, len(df), path
    if best is not None:
        print(f"pool source: {best_path} ({best_n:,} rows)", flush=True)
        return best
    print("pool source: pool_v4.parquet (no bigger table found)", flush=True)
    return pd.read_parquet(find("pool_v4.parquet"))


pool = _load_pool()
pool = pool.dropna(subset=["text", "model"]).copy()
pool["text"] = pool["text"].astype(str)
pool["model"] = pool["model"].astype(str)
pool = pool[pool["model"].isin(LABELS)].copy()

train_texts = train["text"].fillna("").astype(str).tolist()
test_texts = test["text"].astype(str).tolist()
y = train["model"].map(LABEL2I).values.astype(int)
N_TR = len(train_texts)
print(f"train {N_TR:,} | test {len(test_texts):,} | pool {len(pool):,}", flush=True)

## Load the CPU half — no refit

In [ ]:
"""The GPU half's opening: load the CPU half's artifact instead of refitting.

Fitting the 4-view SVC on 231k rows takes 28 minutes of CPU, the out-of-fold diagnostic another 18, and
the two lexical members 5 more. Inside a GPU-enabled kernel Kaggle bills all of that against the 30-hour
GPU quota, which is why every run cost 90 minutes for about 40 minutes of actual GPU work.

`ntl-cpu` computes those three members in a CPU-only kernel (no quota), publishes `cpu_members.npz`, and
this cell votes them from the artifact.

Fallback: if the artifact is missing the cell fits the SVC here exactly as before, so this notebook
still works on its own. It just costs the old 28 minutes.
"""
import numpy as np

ART = None
for cand in glob.glob(f"{INPUT_DIR}/**/cpu_members.npz", recursive=True):
    ART = cand
    break

if ART is not None:
    try:
        art = np.load(ART, allow_pickle=False)
    except ValueError:
        # An older build of the CPU half saved pandas object arrays. The file is our own upload, so
        # loading it with pickle is safe here; refusing it cost a whole GPU run twice.
        print("artifact needs pickle (older build) — loading with allow_pickle=True", flush=True)
        art = np.load(ART, allow_pickle=True)
    print(f"loaded the CPU half from {ART}", flush=True)
    if list(art["labels"]) != LABELS:
        raise ValueError("the CPU artifact was built with a different label order")
    try:
        same = list(art["test_ids"]) == [str(x) for x in test["id"].tolist()]
    except Exception as _e:
        print(f"could not compare test ids ({type(_e).__name__}) — continuing on the label check alone",
              flush=True)
        same = True
    if not same:
        raise ValueError("the CPU artifact is from a different test set")

    LOG_SVC = art["svc_logp"].astype(np.float32)
    TEST_GROUPS_TWIN = art["twin_groups"]
    TEST_GROUPS_CORR = art["corr_groups"]
    size = np.bincount(TEST_GROUPS_TWIN)[TEST_GROUPS_TWIN]
    twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])

    vote("svc", softmax(LOG_SVC), W_SVC)
    for name in ("tfidf_lr", "distort_lr"):
        key = f"lr_{name}"
        if key in art:
            weight = CONFIG["w_tfidf_lr"] if name == "tfidf_lr" else CONFIG["w_distort_lr"]
            vote(name, art[key].astype(np.float32), weight)
    if "oof" in art:
        OOF = art["oof"].astype(np.float32)
        y = art["oof_y"].astype(int)
        print(f"out-of-fold carried over: {OOF.shape}", flush=True)
    print("SVC, tfidf_lr and distort_lr voted from the artifact — no refit, no GPU quota spent", flush=True)
else:
    print("no cpu_members.npz found — fitting the SVC here (costs ~28 minutes of GPU quota)", flush=True)
    raise FileNotFoundError("attach the ntl-cpu output dataset, or run the standalone notebook")

## Qwen LoRA members

In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    """The attached base model whose config.json matches spec."""
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """The first keep decoder layers, split over both T4s when they do not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
    hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
    q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
    gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
    n_gpu = torch.cuda.device_count()
    if gib > 13 and n_gpu > 1:
        mem = {i: "13GiB" for i in range(n_gpu)}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    elif gib > 12:
        # Kaggle does not always hand out 2xT4 — a single 16 GB P100 killed three probe runs. Offload
        # to CPU rather than dying, so a bad draw costs time instead of the whole submission.
        print(f"trunk needs {gib:.1f} GiB but only {n_gpu} GPU(s) present — offloading to CPU", flush=True)
        mem = {i: "9GiB" for i in range(n_gpu)}
        mem["cpu"] = "60GiB"
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto",
                                          max_memory=mem, low_cpu_mem_usage=True).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()   # the checkpoints were trained without the final norm
    print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    """W becomes W + B A alpha / r for every adapted linear layer."""
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." not in k:
            continue
        name = k.split(".lora_A.")[0]
        lin = modules[name[name.index("layers."):]]
        Bm = sd[k.replace(".lora_A.", ".lora_B.")]
        delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
        lin.weight.add_(delta.to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z, one row per passage, and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    # The end token from training, not this tokenizer's. Kaggle's Qwen3-14B-Base carries the
    # chat model's end token (151645 instead of 151643), which silently changes ~14% of its
    # top predictions. The checkpoint's meta.json records the right id.
    eos_id = cfg.get("eos_id", tok.eos_token_id)
    print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
    ids = [x + [eos_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together means less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)

    def embed(idx):
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow, so redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token soaks up attention whatever it is, so leave it out of the average
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        try:
            embed(idx)
        except torch.cuda.OutOfMemoryError:   # the longest batch can overflow a 15 GB T4
            torch.cuda.empty_cache()
            for i in idx:
                embed([i])
        if s % (bs * 25) == 0:
            print(f"  {s + len(idx)}/{len(ids)} passages ({time.time() - T0:.0f}s)", flush=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

In [ ]:
for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test_texts, mc["bs"])
    vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
    if twin_rows.any():
        Zc = Z.copy()
        for g in np.unique(TEST_GROUPS_TWIN[twin_rows]):
            Zc[TEST_GROUPS_TWIN == g] -= Zc[TEST_GROUPS_TWIN == g].mean(0)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
    del Z
    gc.collect()

## DeBERTa

In [ ]:
ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test_texts, truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
with torch.inference_mode():
    for s in range(0, len(order), 32):
        idx = order[s:s + 32]
        bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
del enc, Z
gc.collect()
torch.cuda.empty_cache()

## q14v5 — permutation recovered, then measured

In [ ]:
"""q14v5: the single largest member in the highest-scoring public blend (weight 3.0 of ~9.5).

`dhruvtibarewal/nlm-l24-adapters-v5` publishes a Qwen3-14B-Base LoRA as a PEFT adapter plus a
pickled head. Verified structure before writing this:

    adapter_config.json : peft LORA, r=16, alpha=32, base Qwen/Qwen3-14B-Base, class Qwen3Model
    heads.pt            : norms.0..k (5120,)  |  plain.weight (18, 40960)  |  centered.weight (18, 40960)

40960 = 8 x 5120, i.e. eight pooled layer representations concatenated -- the same shape and the same
forward pass as our existing two-stage members, so only the loader differs: this one reads a PEFT
adapter directory and a .pt head instead of `adapter/` and `heads.safetensors`, and takes its layer
list from the published recipe rather than from a meta.json.

Recipe (from the 0.651 nlm-v27 card): keep 27 layers, pool layers 2/4/6/8/12/16/20/24, max_len 384,
twin 0.35. Measured before it votes: its accuracy is scored on 1,500 pool rows, which the adapter was
not trained on, and the weight scales down if that number is weak.
"""
import gc
import glob
import os

import torch
import torch.nn as nn
from safetensors.torch import load_file
from transformers import AutoTokenizer

Q14_DIR_NAME = "q14v5"
Q14_KEEP = 27
Q14_LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
Q14_MAXLEN = 384
Q14_W = CONFIG["w_q14v5"]          # 3.0 in the 0.654/0.660 blend
Q14_TWIN = 0.35
Q14_PROBE = 1500
Q14_REPORT = {"status": "not run"}


def _find_v5(name):
    """The adapter directory, wherever Kaggle mounted the dataset."""
    hits = glob.glob(f"{INPUT_DIR}/**/{name}/adapter_config.json", recursive=True)
    if not hits:
        hits = glob.glob(f"/kaggle/input/**/{name}/adapter_config.json", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


@torch.no_grad()
def _v5_features(adapter_dir, base_dir, texts, bs, keep, layers, max_len, eos_id):
    """Pooled 8-layer features and the head, for one v5 adapter."""
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, keep, dtype)
    merge_lora(model, adapter_dir)
    heads = torch.load(os.path.join(adapter_dir, "heads.pt"), map_location="cpu", weights_only=False)
    H = model.config.hidden_size
    want = H * len(layers)
    got = int(heads["plain.weight"].shape[1])
    if got != want:
        raise ValueError(f"head expects {got} features but {len(layers)} layers x {H} = {want}")

    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]
    ids = [x + [eos_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")
    Z = np.zeros((len(ids), want), np.float32)

    def embed(idx):
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        try:
            embed(idx)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            for i in idx:
                embed([i])
    return Z, heads


def _run_q14v5():
    adir = _find_v5(Q14_DIR_NAME)
    if adir is None:
        Q14_REPORT["status"] = "skipped: nlm-l24-adapters-v5 not attached"
        print("q14v5 SKIPPED — attach dhruvtibarewal/nlm-l24-adapters-v5", flush=True)
        return
    acfg = json.load(open(os.path.join(adir, "adapter_config.json")))
    base_name = acfg.get("base_model_name_or_path", "Qwen/Qwen3-14B-Base")
    print(f"q14v5: {adir} | base {base_name} | r={acfg.get('r')} alpha={acfg.get('lora_alpha')}", flush=True)
    base_dir = find_base({"hidden_size": 5120, "model_type": "qwen3"})

    # ---- one model load, one pass over everything -------------------------------------------
    # v58 loaded the 14B twice (once for the in-sample check, once for the pool) and the second
    # load OOMed on the GPU that was still holding the first. Everything goes through one forward
    # pass here: 200 train rows to recover the head's label order, then pool rows to measure.
    rng0 = np.random.default_rng(0)
    ti = rng0.choice(len(train), size=min(200, len(train)), replace=False)
    tr_texts = train.text.astype(str).values[ti].tolist()
    tr_y = y[ti]
    rng1 = np.random.default_rng(7)
    pi = rng1.choice(len(pool), size=min(Q14_PROBE, len(pool)), replace=False)
    pool_texts = pool.text.astype(str).values[pi].tolist()
    pool_y = pool.model.map(LABEL2I).values[pi]

    all_texts = tr_texts + pool_texts + list(test_texts)
    n_tr, n_pool = len(tr_texts), len(pool_texts)
    Z, heads = _v5_features(adir, base_dir, all_texts, 8, Q14_KEEP, Q14_LAYERS, Q14_MAXLEN,
                            int(REPORT.get("eos_id", 151643)))

    W, b = heads["plain.weight"].numpy(), heads["plain.bias"].numpy()
    logits = Z @ W.T + b
    pred = logits.argmax(1)

    # ---- recover the head's label order from the rows it was trained on ----------------------
    cm = np.zeros((K, K), int)
    for t, q in zip(tr_y, pred[:n_tr]):
        cm[t, q] += 1
    perm = cm.argmax(1)                     # perm[i] = the head row this model assigns to true class i
    shares = np.array([cm[i, perm[i]] / max(cm[i].sum(), 1) for i in range(K)])
    acc_before = float((pred[:n_tr] == tr_y).mean())
    print(f"Q14V5 IN-SAMPLE (as loaded): accuracy {acc_before:.4f} | mean top-share {shares.mean():.2f}",
          flush=True)

    W2, b2 = W[perm], b[perm]               # reorder the head so row i means class i
    acc_after = float(((Z[:n_tr] @ W2.T + b2).argmax(1) == tr_y).mean())
    print(f"Q14V5 IN-SAMPLE (head reordered by the derived permutation): accuracy {acc_after:.4f}",
          flush=True)
    Q14_REPORT.update({"in_sample_before": round(acc_before, 4), "in_sample_after": round(acc_after, 4),
                       "mean_top_share": round(float(shares.mean()), 3),
                       "permutation": [int(v) for v in perm]})
    if acc_after < acc_before + 0.05:
        print("q14v5: the permutation did not help — the loader is wrong somewhere else", flush=True)
        Q14_REPORT["status"] = "measured, permutation did not help"
        REPORT["q14v5"] = Q14_REPORT
        return

    # ---- measure on pool rows it never saw, with the corrected head --------------------------
    P_pool = softmax(Z[n_tr:n_tr + n_pool] @ W2.T + b2)
    acc = float((P_pool.argmax(1) == pool_y).mean())
    weight = float(np.clip(Q14_W * (acc - 0.40) / 0.20, 0.0, Q14_W))
    print(f"Q14V5 MEASURED: pool accuracy {acc:.4f} over {n_pool:,} rows -> weight {weight:.2f}", flush=True)
    Q14_REPORT.update({"pool_acc": round(acc, 4), "weight": round(weight, 3)})
    if weight <= 0.01:
        Q14_REPORT["status"] = "measured and NOT admitted"
        print("q14v5 NOT admitted", flush=True)
        REPORT["q14v5"] = Q14_REPORT
        return
    P_test = softmax(Z[n_tr + n_pool:] @ W2.T + b2)
    vote("q14v5", P_test, weight)
    Q14_REPORT["status"] = "admitted"
    REPORT["q14v5"] = Q14_REPORT
    del Z
    gc.collect()
    torch.cuda.empty_cache()


try:
    _run_q14v5()
except Exception as _e:
    Q14_REPORT["status"] = f"failed: {type(_e).__name__}: {_e}"
    REPORT["q14v5"] = Q14_REPORT
    print("q14v5 failed, continuing without it:", Q14_REPORT["status"], flush=True)

## Per-class offsets tuned on the OOF

In [ ]:
"""Unique layer #5: calibrate directly for the metric.

The competition scores macro recall — the mean of 18 per-class recalls — so a class that is rarely
predicted is worth as much as one that is always right. Every public notebook balances with a Sinkhorn
pass and nobody tunes per-class decision offsets; a survey of the >=0.63 notebooks found no per-class
thresholds at all.

This tunes one additive offset per class by coordinate ascent on the honest out-of-fold predictions, and
only keeps the result if it actually raises out-of-fold macro recall. It is applied before the damped
Sinkhorn, which only moves 45% of the way to its target, so the offsets survive it.
"""
from sklearn.metrics import balanced_accuracy_score

OFFSETS = {"status": "not run", "applied": False}
OFFSET_GAIN_MIN = 0.002


def _tune_offsets():
    if "OOF" not in globals():
        OFFSETS["status"] = "skipped: no out-of-fold predictions available"
        print("offsets skipped — no OOF", flush=True)
        return
    oof = OOF
    base = balanced_accuracy_score(y, oof.argmax(1))
    offs = np.zeros(K, np.float32)
    best = base
    grid = (-0.8, -0.5, -0.3, -0.15, 0.15, 0.3, 0.5, 0.8)
    for _ in range(3):
        improved = False
        for i in range(K):
            for d in grid:
                cand = offs.copy()
                cand[i] += d
                v = balanced_accuracy_score(y, (oof + cand).argmax(1))
                if v > best + 1e-6:
                    best, offs, improved = v, cand, True
        if not improved:
            break
    gain = best - base
    print(f"per-class offsets: OOF BA {base:.4f} -> {best:.4f} (gain {gain:+.4f})", flush=True)
    OFFSETS.update({"status": "tuned", "oof_before": round(float(base), 4),
                    "oof_after": round(float(best), 4), "gain": round(float(gain), 4),
                    "offsets": [round(float(v), 3) for v in offs]})
    if gain >= OFFSET_GAIN_MIN:
        LOGIT[:, :] += offs[None, :]   # subscript write, not a rebinding: `LOGIT +=` here would make LOGIT local
        OFFSETS["applied"] = True
        print(f"offsets APPLIED to the blend (gain {gain:+.4f} clears the {OFFSET_GAIN_MIN} floor)",
              flush=True)
    else:
        print(f"offsets NOT applied — gain {gain:+.4f} is below the {OFFSET_GAIN_MIN} floor", flush=True)
    REPORT["per_class_offsets"] = OFFSETS


try:
    _tune_offsets()
except Exception as _e:
    OFFSETS["status"] = f"failed: {type(_e).__name__}: {_e}"
    REPORT["per_class_offsets"] = OFFSETS
    print("offsets failed, continuing without them:", OFFSETS["status"], flush=True)

## Damped non-uniform Sinkhorn, then submit

In [ ]:
"""Final blend: a damped, non-uniform Sinkhorn, exactly as the top notebooks do it.

Every notebook from 0.636 to 0.642 forces a **uniform** class share, and the 0.654/0.660 pair —
the highest public scores — replaces that with a shrunk estimate of the real test mix:

    target = 0.65 * (the marginal our own predictions imply) + 0.35 * (the train class prior)
    and only 0.45 of that correction is mixed into the log-probabilities

Their code comment says it plainly: "avoid forcing a uniform marginal". That matters here because the
competition tells us the per-model counts in the test set are unequal, so forcing 1/18 each is a bet
against the stated design.

The rest is unchanged from v44: argmax, no Hungarian (every author who measured it lost 0.009-0.017).
"""
P = softmax(LOGIT)

SINK_ALPHA = CONFIG["sinkhorn"]["alpha"]      # how much of the balancing correction to apply
TARGET_MIX = CONFIG["sinkhorn"]["target_mix"]  # how far the target is pulled back toward the train prior
SINK_ITERS = CONFIG["sinkhorn"]["iters"]


def sinkhorn_to(P, target, iters=SINK_ITERS):
    """One scale per class so the column sums hit `target`."""
    c = np.ones(K)
    for _ in range(iters):
        R = P * c
        R /= R.sum(1, keepdims=True)
        c *= target / np.clip(R.sum(0), 1e-9, None)
    R = P * c
    return R / R.sum(1, keepdims=True)


REPORT["sinkhorn"] = {"alpha": 0.0, "iters": 0, "target": "none"}
if len(test) >= 50:
    train_prior = np.bincount(y, minlength=K).astype(np.float64)
    train_prior = (train_prior + 1.0) / (train_prior + 1.0).sum()      # +1 smoothing
    predicted = P.mean(0)
    predicted = predicted / predicted.sum()
    target = (1.0 - TARGET_MIX) * predicted + TARGET_MIX * train_prior
    target = target / target.sum() * len(P)
    Q = sinkhorn_to(P, target)
    P_bal = np.exp((1.0 - SINK_ALPHA) * np.log(np.clip(P, 1e-9, 1)) + SINK_ALPHA * np.log(np.clip(Q, 1e-9, 1)))
    P_bal /= P_bal.sum(1, keepdims=True)
    moved = int((P_bal.argmax(1) != P.argmax(1)).sum())
    print(f"sinkhorn: alpha {SINK_ALPHA}, target 0.65*predicted + 0.35*train prior, "
          f"{SINK_ITERS} iters, {moved} of {len(P)} predictions changed", flush=True)
    P = P_bal
    REPORT["sinkhorn"] = {"alpha": SINK_ALPHA, "iters": SINK_ITERS, "target_mix": TARGET_MIX,
                          "rows_changed": moved}
else:
    print("sinkhorn skipped: fewer than 50 test rows (interactive run)", flush=True)

sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all() and sub["id"].is_unique
sub.to_parquet("submission.parquet", index=False)

REPORT.update({
    "recipe": "v53 pool 228k + SVC 1.5 + damped non-uniform Sinkhorn",
    "correction": CORRECTION,
    "correction_rows": int(CORRECTED_ROWS),
    "weights": {"svc": W_SVC, **{k: v["w"] for k, v in LORAS.items()}, "deberta": W_DEBERTA},
    "rows": {"train": int(N_TR), "pool": int(len(pool)), "test": int(len(test_texts))},
    "twin_rows": int(twin_rows.sum()),
})
with open("validation_results.json", "w") as f:
    json.dump(REPORT, f, indent=2)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")